# 00. Official Contract Audit

이번 노트북의 목적은 `2차 현장실험 데이터`를 바로 학습시키는 것이 아니라, 먼저 **CLRKDNet이 요구하는 데이터셋 계약**을 공식 구현 기준으로 고정하는 것이다.

여기서 말하는 계약은 대략 아래 네 가지다.

1. 원본 이미지가 어떤 해상도와 crop 규칙으로 모델 입력에 들어가는가.
2. `.lines.txt` 파일은 어떤 좌표계와 형식을 가져야 하는가.
3. segmentation mask는 어떤 위치에 있어야 하고, train list에는 어떻게 적어야 하는가.
4. CLRKDNet은 차선 검출까지만 담당하므로, 주행 제어는 후처리에서 별도로 만들어야 한다는 경계.

이 작업을 먼저 해두면, 뒤의 HSV sweep이나 pseudo-label 생성이 '내가 임의로 대충 만든 라벨'이 아니라, 공식 loader가 실제로 읽을 수 있는 형식으로 가고 있는지 계속 확인할 수 있다.

## 0. 참고 기준

이번 노트북은 아래 소스를 기준으로 삼는다.

- CLRKDNet 공식 repo: https://github.com/weiqingq/CLRKDNet
- CLRKDNet paper: https://arxiv.org/abs/2405.12503
- CLRNet 공식 repo: https://github.com/Turoad/CLRNet
- CLRNet paper: https://arxiv.org/abs/2203.10350

우리 로컬 workspace에는 공식 repo clone이 이미 들어있으므로, 실제 구현 계약은 로컬 파일을 직접 읽어서 확인한다.

In [1]:
from pathlib import Path
import json
import re
from collections import Counter

ROOT = Path(r"~/02_Projects/University/26-1_EmbeddedArtificialSystemOptimization")
EXP_ROOT = ROOT / "10_experiments" / "05_lane_finetune_dataset_pipeline"
CLRKDNET = ROOT / "00_reference" / "repos" / "CLRKDNet"
CLRNET = ROOT / "00_reference" / "repos" / "CLRNet"
FIELD2 = ROOT / "10_experiments" / "04_field2_map_lab" / "field2_experiment" / "data" / "20260501_field2"

paths = {
    "EXP_ROOT": EXP_ROOT,
    "CLRKDNET": CLRKDNET,
    "CLRNET": CLRNET,
    "FIELD2": FIELD2,
}
for name, path in paths.items():
    print(f"{name:10s} exists={path.exists()}  {path}")
    assert path.exists(), path

EXP_ROOT   exists=True  ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\05_lane_finetune_dataset_pipeline
CLRKDNET   exists=True  ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\00_reference\repos\CLRKDNet
CLRNET     exists=True  ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\00_reference\repos\CLRNet
FIELD2     exists=True  ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\04_field2_map_lab\field2_experiment\data\20260501_field2


## 1. 공식 CULane config에서 고정해야 할 값

CLRKDNet의 `configs/ResNet18_CULane.py`는 CULane 기준 학습 설정이다. 여기서 중요한 값은 모델 구조 전체가 아니라, 데이터셋 빌더가 맞춰야 하는 좌표계다.

- `ori_img_w`, `ori_img_h`: 원본 이미지 좌표계
- `cut_height`: 위쪽을 얼마나 잘라낸 뒤 학습/추론에 넣는지
- `img_w`, `img_h`: 모델 입력 크기
- `num_points`: lane x 좌표를 샘플링하는 row 개수
- `max_lanes`: 한 이미지에서 최대 lane 수
- `sample_y`: evaluation/visualization 쪽에서 쓰는 원본 y sampling 기준

우리 map 데이터는 CULane 원본 해상도와 다르므로, 이 숫자를 그대로 쓰는 것이 아니라 **같은 의미를 우리 카메라 좌표계로 옮겨야 한다**.

In [2]:
config_path = CLRKDNET / "configs" / "ResNet18_CULane.py"
config_text = config_path.read_text(encoding="utf-8")

interesting = [
    "num_points", "max_lanes", "sample_y", "ori_img_w", "ori_img_h",
    "img_w", "img_h", "cut_height", "test_parameters", "dataset_path",
    "diff_path", "threshold"
]
for i, line in enumerate(config_text.splitlines(), start=1):
    stripped = line.strip()
    if any(stripped.startswith(key) for key in interesting):
        print(f"{i:03d}: {line}")

001: num_points = 72
002: max_lanes = 4
003: sample_y = range(589, 230, -20)
021: test_parameters = dict(conf_threshold=0.4, nms_thres=50, nms_topk=max_lanes)
039: ori_img_w = 1640
040: ori_img_h = 590
041: img_w = 800
042: img_h = 320
043: cut_height = 270
091: dataset_path = './data/CULane'
093: diff_path = 'data/CULane/list/train_diffs.npz'
094: threshold = 15


In [3]:
official_culane = {
    "ori_img_w": 1640,
    "ori_img_h": 590,
    "cut_height": 270,
    "img_w": 800,
    "img_h": 320,
    "num_points": 72,
    "max_lanes": 4,
    "sample_y": "range(589, 230, -20)",
}

our_map = {
    "ori_img_w": 1296,
    "ori_img_h": 972,
    "cut_height": 445,
    "img_w": 800,
    "img_h": 320,
    "num_points": 72,
    "max_lanes": 4,
    "sample_y": "range(971, 444, -20)",
}

print("official CULane")
print(json.dumps(official_culane, indent=2, ensure_ascii=False))
print("\nour map draft")
print(json.dumps(our_map, indent=2, ensure_ascii=False))

visible_h = our_map["ori_img_h"] - our_map["cut_height"]
print(f"\n우리 crop 이후 visible height = {visible_h}px")
print(f"모델 입력 resize = {our_map['img_w']}x{our_map['img_h']}")

official CULane
{
  "ori_img_w": 1640,
  "ori_img_h": 590,
  "cut_height": 270,
  "img_w": 800,
  "img_h": 320,
  "num_points": 72,
  "max_lanes": 4,
  "sample_y": "range(589, 230, -20)"
}

our map draft
{
  "ori_img_w": 1296,
  "ori_img_h": 972,
  "cut_height": 445,
  "img_w": 800,
  "img_h": 320,
  "num_points": 72,
  "max_lanes": 4,
  "sample_y": "range(971, 444, -20)"
}

우리 crop 이후 visible height = 527px
모델 입력 resize = 800x320


## 2. CULane loader가 실제로 읽는 파일

공식 `CULane` dataset loader는 아래 구조를 기대한다.

```text
dataset_root/
  driver_xx_xxframe/.../*.jpg
  driver_xx_xxframe/.../*.lines.txt
  laneseg_label_w16/.../*.png
  list/
    train_gt.txt
    val.txt
    test.txt
```

`train_gt.txt`는 보통 `이미지 경로 + mask 경로 + lane 존재 여부`를 담고, `val/test`는 이미지 경로만 있어도 loader가 같은 위치의 `.lines.txt`를 찾아 읽는다.

중요한 점: `.lines.txt`는 최종 모델 입력 크기인 800x320 좌표가 아니라, **원본 이미지 좌표계에서의 lane point**를 담아야 한다. 이후 `BaseDataset`이 `cut_height`만큼 y를 빼고, `GenerateLaneLine`이 resize/augmentation 후 내부 label tensor로 변환한다.

In [4]:
def print_matching_lines(path: Path, needles, context=0):
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    hits = []
    for idx, line in enumerate(lines, start=1):
        if any(n in line for n in needles):
            lo = max(1, idx - context)
            hi = min(len(lines), idx + context)
            for j in range(lo, hi + 1):
                hits.append((j, lines[j-1]))
            hits.append((None, ""))
    print(f"# {path.relative_to(CLRKDNET)}")
    for no, line in hits:
        if no is None:
            print()
        else:
            print(f"{no:03d}: {line}")

print_matching_lines(
    CLRKDNET / "clrkd" / "datasets" / "culane.py",
    ["LIST_FILE", "train_gt.txt", "mask_path", "anno_path", "lines.txt", "lane_exist"],
    context=1,
)

# clrkd\datasets\culane.py
011: 
012: LIST_FILE = {
013:     'train': 'list/train_gt.txt',

012: LIST_FILE = {
013:     'train': 'list/train_gt.txt',
014:     'val': 'list/val.txt',

034:         super().__init__(data_root, split, processes=processes, cfg=cfg)
035:         self.list_path = osp.join(data_root, LIST_FILE[split])
036:         self.split = split

096:             mask_line = mask_line[1 if mask_line[0] == '/' else 0::]
097:             mask_path = os.path.join(self.data_root, mask_line)
098:             infos['mask_path'] = mask_path

097:             mask_path = os.path.join(self.data_root, mask_line)
098:             infos['mask_path'] = mask_path
099: 

101:             exist_list = [int(l) for l in line[2:]]
102:             infos['lane_exist'] = np.array(exist_list)
103: 

103: 
104:         anno_path = img_path[:-3] + 'lines.txt'  # remove sufix jpg and add lines.txt
105:         with open(anno_path, 'r') as anno_file:

104:         anno_path = img_path[:-3] + 'lines

In [5]:
print_matching_lines(
    CLRKDNET / "clrkd" / "datasets" / "base_dataset.py",
    ["cut_height", "mask_path", "new_lanes", "lanes.append"],
    context=2,
)

# clrkd\datasets\base_dataset.py
038:         data_info = self.data_infos[idx]
039:         img = cv2.imread(data_info['img_path'])
040:         img = img[self.cfg.cut_height:, :, :]
041:         sample = data_info.copy()
042:         sample.update({'img': img})

043: 
044:         if self.training:
045:             label = cv2.imread(sample['mask_path'], cv2.IMREAD_UNCHANGED)
046:             if len(label.shape) > 2:
047:                 label = label[:, :, 0]

047:                 label = label[:, :, 0]
048:             label = label.squeeze()
049:             label = label[self.cfg.cut_height:, :]
050:             sample.update({'mask': label})
051: 

050:             sample.update({'mask': label})
051: 
052:             if self.cfg.cut_height != 0:
053:                 new_lanes = []
054:                 for i in sample['lanes']:

051: 
052:             if self.cfg.cut_height != 0:
053:                 new_lanes = []
054:                 for i in sample['lanes']:
055:              

## 3. `.lines.txt`가 내부 label tensor로 바뀌는 방식

`GenerateLaneLine`은 `.lines.txt`의 좌표 점들을 곡선으로 보고, 모델 입력 크기에서 72개 row에 대해 x 좌표를 샘플링한다.

이 말은 pseudo-label 생성기가 해야 할 일이 명확하다는 뜻이다.

1. raw image에서 노란선 후보를 찾는다.
2. ego-lane의 좌/우 차선에 해당하는 곡선 point들을 만든다.
3. 이 point들을 원본 이미지 좌표계의 `.lines.txt`로 저장한다.
4. train용 segmentation mask도 같은 raw 좌표계에 맞춰 저장한다.
5. 공식 loader가 crop/resize/augmentation/label tensor 생성을 맡게 한다.

즉, 우리가 05에서 만들 것은 모델 label tensor가 아니라, 공식 loader가 label tensor를 만들 수 있도록 하는 **원본 좌표 pseudo annotation**이다.

In [6]:
print_matching_lines(
    CLRKDNET / "clrkd" / "datasets" / "process" / "generate_lane_line.py",
    ["num_points", "n_offsets", "offsets_ys", "sample_lane", "normalize", "lanes = np.ones", "lane_line"],
    context=2,
)

# clrkd\datasets\process\generate_lane_line.py
016:         self.transforms = transforms
017:         self.img_w, self.img_h = cfg.img_w, cfg.img_h
018:         self.num_points = cfg.num_points
019:         self.n_offsets = cfg.num_points
020:         self.n_strips = cfg.num_points - 1

017:         self.img_w, self.img_h = cfg.img_w, cfg.img_h
018:         self.num_points = cfg.num_points
019:         self.n_offsets = cfg.num_points
020:         self.n_strips = cfg.num_points - 1
021:         self.strip_size = self.img_h / self.n_strips

018:         self.num_points = cfg.num_points
019:         self.n_offsets = cfg.num_points
020:         self.n_strips = cfg.num_points - 1
021:         self.strip_size = self.img_h / self.n_strips
022:         self.max_lanes = cfg.max_lanes

021:         self.strip_size = self.img_h / self.n_strips
022:         self.max_lanes = cfg.max_lanes
023:         self.offsets_ys = np.arange(self.img_h, -1, -self.strip_size)
024:         self.training = trainin

## 4. 2차 현장실험 데이터의 현재 구조

이제 공식 계약을 본 뒤, 실제 입력이 될 2차 데이터를 확인한다. 여기서는 이미지 내용을 판단하지 않고, 폴더 구조와 개수만 본다.

학습 후보는 우선 `raw/01_lane_drive_capture`를 중심으로 보되, `debug/`는 사람이 확인하는 overlay/reference로 쓴다.

In [7]:
image_exts = {".jpg", ".jpeg", ".png"}

def count_images(root: Path):
    # Windows/remote-copied files can occasionally behave oddly with Path.is_file().
    # For this inventory step, suffix-based counting is safer and matches PowerShell's file count.
    return sum(1 for p in root.rglob("*") if p.suffix.lower() in image_exts)

for sub in ["raw", "debug"]:
    root = FIELD2 / sub
    print(f"{sub:5s}: {count_images(root)} images")
    for child in sorted(root.iterdir()):
        if child.is_dir():
            print(f"  {child.name:28s} {count_images(child):5d}")
    print()

raw  : 5729 images
  00_camera_pose                   0
  01_lane_drive_capture         2467
  03_lane_preview                 50
  06_sign_capture_drive         2542
  07_traffic_capture             569
  08_redline_capture_debug       101

debug: 5729 images
  00_camera_pose                   0
  01_lane_drive_capture         2467
  03_lane_preview                 50
  06_sign_capture_drive         2542
  07_traffic_capture             569
  08_redline_capture_debug       101



In [8]:
lane_root = FIELD2 / "raw" / "01_lane_drive_capture"
scene_counts = []
for scene_dir in sorted(lane_root.iterdir()):
    if scene_dir.is_dir():
        scene_counts.append((scene_dir.name, count_images(scene_dir)))

print("field2 lane scene distribution")
for scene, count in scene_counts:
    print(f"{scene:24s} {count:5d}")

print(f"\ntotal lane raw images = {sum(c for _, c in scene_counts)}")

field2 lane scene distribution
failure_cases              488
intersection_approach       93
intersection_left          285
intersection_right         287
left_curve                 227
off_center_left            344
off_center_right           455
right_curve                231
straight                    57

total lane raw images = 2467


## 5. 이번 파이프라인에서 반드시 지킬 체크리스트

다음 노트북부터는 아래 체크리스트를 기준으로 진행한다.

- raw 이미지는 1296x972 기준으로 보관한다.
- 학습용 `.lines.txt`의 좌표도 raw 이미지 좌표계 기준으로 저장한다.
- `cut_height=445`는 loader/config에서 적용한다. 라벨 생성 단계에서 임의로 y를 줄여서 저장하지 않는다.
- segmentation mask는 raw 이미지와 같은 크기로 저장하고, loader가 동일하게 crop한다.
- pseudo-label은 `auto_accept`, `needs_review`, `reject`로 나눈다.
- OpenCV HSV는 라벨 생성 도구일 뿐, 최종 아키텍처의 lane inference 대체물이 아니다.
- 최종 산출물은 노트북 결과가 아니라, `src/build_map_culane_dataset.py`로 반복 실행 가능한 빌더여야 한다.

In [9]:
contract = {
    "model_family": "CLRKDNet / CLRNet-style lane detection",
    "task_boundary": "lane curve detection only; control is downstream postprocess",
    "input_raw_size": [1296, 972],
    "cut_height": 445,
    "model_input_size": [800, 320],
    "annotation_coordinate_system": "raw image coordinates before cut_height",
    "required_outputs": [
        "image files",
        ".lines.txt lane point files",
        "segmentation masks for training",
        "list/train_gt.txt",
        "list/val.txt",
        "list/test.txt",
        "review overlays and quality report",
    ],
    "next_notebook": "01_field2_data_inventory.ipynb",
}
print(json.dumps(contract, indent=2, ensure_ascii=False))

{
  "model_family": "CLRKDNet / CLRNet-style lane detection",
  "task_boundary": "lane curve detection only; control is downstream postprocess",
  "input_raw_size": [
    1296,
    972
  ],
  "cut_height": 445,
  "model_input_size": [
    800,
    320
  ],
  "annotation_coordinate_system": "raw image coordinates before cut_height",
  "required_outputs": [
    "image files",
    ".lines.txt lane point files",
    "segmentation masks for training",
    "list/train_gt.txt",
    "list/val.txt",
    "list/test.txt",
    "review overlays and quality report"
  ],
  "next_notebook": "01_field2_data_inventory.ipynb"
}


## 6. 00번 노트북 실행 결과 정리

이번 노트북에서 확인한 핵심은 **2차 현장실험 이미지를 바로 학습시키는 것이 아니라, 먼저 CLRKDNet/CULane loader가 요구하는 데이터 계약을 정확히 맞춰야 한다**는 점임.

공식 `ResNet18_CULane.py` 기준으로 CLRKDNet은 `num_points=72`, `max_lanes=4`, `img_w=800`, `img_h=320` 구조를 사용함. CULane 원본 설정은 `ori_img_w=1640`, `ori_img_h=590`, `cut_height=270`이지만, 우리 프로젝트 카메라는 `1296x972`이고 현재 설계는 `cut_height=445`, 모델 입력은 동일하게 `800x320`으로 둠. 따라서 공식 숫자를 그대로 복사하는 것이 아니라, **공식 처리 의미를 우리 카메라 좌표계로 옮기는 것**이 이번 파이프라인의 출발점임.

공식 loader 확인 결과, `.lines.txt`는 crop 이후 800x320 좌표가 아니라 **raw image 좌표계의 lane point**를 저장해야 함. 이후 `BaseDataset`이 이미지와 mask를 `cut_height`만큼 자르고, lane point의 y 좌표에서도 `cut_height`를 빼며, `GenerateLaneLine`이 resize/augmentation 후 내부 `lane_line` tensor를 생성함. 즉 우리가 직접 모델 label tensor를 만드는 것이 아니라, 공식 loader가 tensor를 만들 수 있도록 원본 좌표 annotation과 mask/list 구조를 맞춰야 함.

2차 현장실험 데이터는 현재 `raw`와 `debug`가 각각 `5729장`으로 잡혔고, 이 중 lane fine-tuning 후보인 `raw/01_lane_drive_capture`는 총 `2467장`임. scene별 분포는 `failure_cases 488`, `intersection_approach 93`, `intersection_left 285`, `intersection_right 287`, `left_curve 227`, `off_center_left 344`, `off_center_right 455`, `right_curve 231`, `straight 57`장으로 확인됨. 이 분포는 단순 직선보다 실패/교차로/오프센터/곡선 케이스가 많아서, 모델을 map 도메인에 맞추기 위한 재료로는 의미가 있음. 다만 `failure_cases`나 교차로 데이터는 자동 pseudo-label 품질이 흔들릴 가능성이 커서, 이후 단계에서 반드시 `auto_accept / needs_review / reject`로 분리해야 함.

따라서 다음 `01_field2_data_inventory.ipynb`에서는 단순 파일 개수보다 한 단계 더 들어가서, 각 scene의 실제 이미지 샘플, 해상도, 색상, 중복/연속 프레임 특성, raw-debug 대응 여부를 확인해야 함. 그 다음에야 HSV threshold sweep과 lane pairing 규칙을 정하는 것이 논리적으로 맞음.